In [ ]:
import sys
from pathlib import Path
HERE_DIR = Path.cwd() 
ROOT_DIR = HERE_DIR.parent.parent
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)
    print(f"Path adapted. Set to: {ROOT_DIR}")

In [ ]:
import os
import json
import regex as re
import torch
import yaml
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from model.ConvLSTM_model import ConvLSTM_Model
# from model.dataset import ARCEME_Dataset
from model.helpers import get_best_model_path, plot_patch_comparison, get_ckpt_and_hparams, prepare_test_loader_from_cfg
from evaluation.Cube_Evaluator import CubeEvaluator

## Get best model

In [ ]:
best_ckpt_new = get_best_model_path(log_dir=os.path.join(ROOT_DIR, "model","tb_logs"))
best_ckpt_old = get_best_model_path(log_dir=os.path.join(ROOT_DIR, "..", "cascading_extremes_recovery_prediction/ConvLSTM/tb_logs"))
info_new = get_ckpt_and_hparams(best_ckpt_new)
info_old = get_ckpt_and_hparams(best_ckpt_old)
print(f"Best old model: {best_ckpt_old}")
print(f"Bestes new model: {best_ckpt_new}")

## Get config

In [ ]:
with open(info_new["hparams"], "r") as f:
    cfg_new = yaml.safe_load(f)
cfg_new = cfg_new["cfg"]
patch_size = cfg_new["data"]["patch_size"]
context_length = cfg_new["data"]["context_length"]
target_length = cfg_new["data"]["target_length"]
v_cfg_new = cfg_new["data"]["variables"]

In [ ]:
with open(info_old["hparams"], "r") as f:
    cfg_old = yaml.safe_load(f)
cfg_old = cfg_old["cfg"]
patch_size = cfg_old["data"]["patch_size"]
context_length = cfg_old["data"]["context_length"]
target_length = cfg_old["data"]["target_length"]
v_cfg_old = cfg_old["data"]["variables"]

In [ ]:
cfg_old

## Initialize model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == "cpu":
    model_new = ConvLSTM_Model.load_from_checkpoint(best_ckpt_new, map_location=device)
else:
    model_new = ConvLSTM_Model.load_from_checkpoint(best_ckpt_new)
model_new.to(device) 

## Get test data

In [ ]:
for name, p in model_new.model.encoder_cells.named_parameters():
    print(name, p.shape)
# hidden_h und hidden_C: 1 (batchsize), 64 == hidden_dim, 1 = H & 1=W(wird später auf patch_size aufgeblasen)
# weight: 256 (4 * 64 = Anzahl Gates (i,f,o,g) * hidden_dim), 84 = 20 + 64 (input_dim (variablen) + hidden_dim)), 5 5 : kernel größe
# bias: 4 * 64: 4 Gates * hidden_dim
# weight (1): 256: Gates *  hidden_dim, 128: input_dim (prev_hidden_dim) + hidden_dim = 64 + 64

In [ ]:
match = re.search(r"(fold_\d+)", best_ckpt_new)
fold = match.group(1)
cv_split_path = os.path.join(Path(best_ckpt_new).parent.parent.parent, "cv_splits.json")
with open(cv_split_path) as f:
    d = json.load(f)
val_files = d[fold]["val_files"]

In [ ]:
DATA_DIR = "/scratch/sloeblein/postprocessed"
val_files = [
    "2018-0376-PRT_postprocessed.zarr",
    "2022-0445-SDN_postprocessed.zarr",
    "2018-0103-BWA_postprocessed.zarr"]
for i, file in enumerate(val_files):
    full_path = os.path.join(DATA_DIR, file)
    val_files[i] = full_path

In [ ]:
# Configuration
N_CUBES_TO_LOAD = len(val_files)
PATCHES_PER_CUBE = 16

test_loader, eval_files = prepare_test_loader_from_cfg(
    val_files, 
    cfg=cfg_new,
    n_cubes=N_CUBES_TO_LOAD, 
    patches_per_cube=PATCHES_PER_CUBE
)

In [ ]:
cube = CubeEvaluator(model_new, device, cfg_new["data"]["target_length"], cfg_new["data"]["context_length"])
all_cube_results = cube.run(test_loader, eval_files, PATCHES_PER_CUBE, False)

In [ ]:
def plot_global_evaluation(all_cube_results):
    """
    Summarizes all cubes, calculates overall weighted stats, 
    and breaks down performance per timestep.
    """
    if not all_cube_results:
        print("No results to evaluate.")
        return

    # --- 1. CUBE-LEVEL DATA ---
    cube_rows = []
    for i, res in enumerate(all_cube_results):
        valid_steps = [m for m in res if m['num_valid'] > 0]
        if not valid_steps: continue

        cube_rows.append({
            "Cube_ID": i,
            "MSE_Model": np.mean([m['mse_pred'] for m in valid_steps]),
            "MSE_Base": np.mean([m['mse_base'] for m in valid_steps]),
            "MAE_Model": np.mean([m['mae_pred'] for m in valid_steps]),
            "Bias": np.mean([m['bias_delta'] for m in valid_steps]),
            "Valid_Pixels": np.sum([m['num_valid'] for m in valid_steps])
        })

    df_cubes = pd.DataFrame(cube_rows)
    
    # --- 2. TIMESTEP-LEVEL DATA (Evolution) ---
    # We aggregate steps across all cubes
    step_rows = []
    for t in range(5):  # Assuming 5 timesteps
        t_metrics = [cube[t] for cube in all_cube_results if cube[t]['num_valid'] > 0]
        if not t_metrics: continue
        
        step_rows.append({
            "Step": f"T+{t+1}",
            "MSE_Model": np.mean([m['mse_pred'] for m in t_metrics]),
            "MSE_Base": np.mean([m['mse_base'] for m in t_metrics]),
            "MAE_Model": np.mean([m['mae_pred'] for m in t_metrics]),
            "Bias_Delta": np.mean([m['bias_delta'] for m in t_metrics])
        })
    df_steps = pd.DataFrame(step_rows)

    # --- 3. OVERALL CALCULATION ---
    total_valid = df_cubes["Valid_Pixels"].sum()
    overall_row = {
        "Cube_ID": "✨ OVERALL",
        "MSE_Model": np.average(df_cubes["MSE_Model"], weights=df_cubes["Valid_Pixels"]),
        "MSE_Base": np.average(df_cubes["MSE_Base"], weights=df_cubes["Valid_Pixels"]),
        "MAE_Model": np.average(df_cubes["MAE_Model"], weights=df_cubes["Valid_Pixels"]),
        "Bias": np.average(df_cubes["Bias"], weights=df_cubes["Valid_Pixels"]),
        "Valid_Pixels": total_valid
    }
    df_cubes = pd.concat([df_cubes, pd.DataFrame([overall_row])], ignore_index=True)
    df_cubes["Improvement_%"] = (1 - (df_cubes["MSE_Model"] / df_cubes["MSE_Base"])) * 100

    # --- OUTPUT ---
    print("\n" + "="*40 + " 📊 GLOBAL SUMMARY " + "="*40)
    display(df_cubes.round(6))
    
    print("\n" + "="*40 + " ⏳ TEMPORAL EVOLUTION " + "="*40)
    display(df_steps.round(6))

    # --- PLOTS ---
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6))
    
    # Plot 1: MSE per Cube
    plot_df = df_cubes[df_cubes["Cube_ID"] != "✨ OVERALL"]
    x = np.arange(len(plot_df))
    ax1.bar(x - 0.2, plot_df["MSE_Model"], 0.4, label='Model', color='skyblue')
    ax1.bar(x + 0.2, plot_df["MSE_Base"], 0.4, label='Baseline', color='lightgray')
    ax1.set_title("MSE per Cube")
    ax1.set_xticks(x); ax1.set_xticklabels(plot_df["Cube_ID"])
    ax1.legend()

    # Plot 2: Relative Improvement
    colors = ['green' if v > 0 else 'red' for v in plot_df["Improvement_%"]]
    ax2.bar(plot_df["Cube_ID"], plot_df["Improvement_%"], color=colors)
    ax2.axhline(0, color='black', linewidth=1)
    ax2.set_title("Improvement % vs Baseline")

    # Plot 3: Evolution of Metrics (MSE & Bias)
    ax3.plot(df_steps["Step"], df_steps["MSE_Model"], 'r-o', label='MSE Model')
    ax3.plot(df_steps["Step"], df_steps["MSE_Base"], 'k--', label='MSE Base')
    ax3_twin = ax3.twinx()
    ax3_twin.bar(df_steps["Step"], df_steps["Bias_Delta"], alpha=0.2, color='purple', label='Bias')
    ax3.set_title("Error & Bias Evolution over Time")
    ax3.legend(loc='upper left'); ax3_twin.legend(loc='upper right')

    plt.tight_layout(); plt.show()
    return df_cubes, df_steps

In [ ]:
# After the run is finished:
final_stats = plot_global_evaluation(all_cube_results)

In [ ]:
CSV_PATH = "../../final_processing_pipeline/data/train_test_split.csv"
df = pd.read_csv(CSV_PATH)
df[df["DisNo."]=="2018-0103-BWA"][["pheno_season_name", "koppen_geiger" ]]